# Load data

In [ ]:
import numpy as np
import pandas as pd

# df = pd.read_parquet("training_data/rdbms_all_methods_training_data.parquet")
df = pd.read_parquet("training_data/rdbms_training_data.parquet")
# rename min_cut_upper to min_cut
df.rename(columns={"min_cut_upper": "min_cut"}, inplace=True)
print(df.columns.tolist())

In [ ]:
SQL_FEATURES = [
    "num_joins",
    "num_and_clauses",
    "num_select_columns",
    "num_agg_funcs",
    "num_where_clauses",
    "num_eq_predicates",
]

GRAPH_FEATURES = [
    "edge_count",
    "max_degree",
    "min_cut",
    "diameter",
    "radius",
    "average_degree",
    "average_clustering_coefficient",
    "average_shortest_path_length",
    "central_point_of_dominance",
    "std_dev_adjacency_matrix",
]

STATIC_FEATURES = [
    "num_qubits",
    "width",
    "depth",
    "circuit_size",
    "pauli_gate_count",
    "two_qubit_gate_count",
    "two_qubit_gate_percentage",
    "locality_ratio",
    "idling_score",
    "density_score",
]

DYNAMIC_FEATURES = [
    "statevector_saved_sparsity",
    "statevector_saved_shannon_entropy",
    # "statevector_saved_von_neumann_entropy",
]

ALL_FEATURES = SQL_FEATURES + GRAPH_FEATURES + STATIC_FEATURES + DYNAMIC_FEATURES

# BUG FIX< statevector saved part of statevector...>

In [ ]:
df.drop(columns=[c for c in df.columns if "statevector_saved_e" in c or "statevector_saved_m" in c], inplace=True)

# Construct y target variable

In [ ]:
RDBMS_METHODS = ["sqlite", "ducksql","psql",]
QISKIT_METHODS = ["density_matrix", "matrix_product_state", "extended_stabilizer", "statevector",]


ALL_METHODS = RDBMS_METHODS + QISKIT_METHODS
# concatenate for all m in ALL_METHODS, the columns that contain m and end with time_s or execution_time
timecols = []
memcols = []
for m in ALL_METHODS:
    timecols.extend([c for c in df.columns if m in c and c.endswith(("time_s", "execution_time"))])
    memcols.extend([c for c in df.columns if m in c and c.endswith(("memory_usage", "mb"))])
print("Time columns:", timecols)
print("Memory columns:", memcols)

In [ ]:
# Build method -> column mapping
time_cols = {
    m: [c for c in timecols if m in c]
    for m in ALL_METHODS
}

mem_cols = {
    m: [c for c in memcols if m in c]
    for m in ALL_METHODS
}

# Remove methods that actually have no columns
time_cols = {m: cols for m, cols in time_cols.items() if len(cols) > 0}
mem_cols  = {m: cols for m, cols in mem_cols.items() if len(cols) > 0}

# Vectorized min-per-method
time_min_df = pd.DataFrame({
    m: df[cols].min(axis=1)
    for m, cols in time_cols.items()
})

mem_min_df = pd.DataFrame({
    m: df[cols].min(axis=1)
    for m, cols in mem_cols.items()
})

# Best method per row
df["best_time_method"] = time_min_df.idxmin(axis=1)
df["best_mem_method"]  = mem_min_df.idxmin(axis=1)

TARGETS = ["best_time_method", "best_mem_method"]

df = df[ALL_FEATURES + TARGETS]

print(f"DataFrame shape: {df.shape} with columns: {df.columns.tolist()}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, target in enumerate(TARGETS):
    ax = axes[i]
    sns.countplot(x=target, data=df, ax=ax)

    ax.set_title(f"Distribution of {target}")
    ax.set_xlabel("Method")
    ax.set_ylabel("Count")

    total = len(df)

    # Add percentage labels
    for p in ax.patches:
        count = p.get_height()
        percentage = f"{100 * count / total:.1f}%"
        x = p.get_x() + p.get_width() / 2
        y = count

        ax.text(x, y + total * 0.005, percentage,
                ha="center", va="bottom", fontsize=10)

    # Add headroom so labels don't overflow
    ymax = max(p.get_height() for p in ax.patches)
    ax.set_ylim(0, ymax * 1.15)

    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
#Lets now replace the TARGETS wit a binary classification of whether the best method is RDBMS or QISKIT. 

print("Unique values in best_time_method before:", df["best_time_method"].unique())
print("Unique values in best_mem_method before:", df["best_mem_method"].unique())

df["best_time_method"] = df["best_time_method"].apply(lambda m: "RDBMS" if m in RDBMS_METHODS else "QISKIT")
df["best_mem_method"]  = df["best_mem_method"].apply(lambda m: "RDBMS" if m in RDBMS_METHODS else "QISKIT")

print("Unique values in best_time_method before:", df["best_time_method"].unique())
print("Unique values in best_mem_method before:", df["best_mem_method"].unique())


# Let us plot a histogram of the new binary targets.
# How many times is RDBMS the best time method vs QISKIT? And same for memory.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, target in enumerate(TARGETS):
    ax = axes[i]
    sns.countplot(x=target, data=df, ax=ax)

    ax.set_title(f"Distribution of {target} (Binary)")
    ax.set_xlabel("Method Type")
    ax.set_ylabel("Count")

    total = len(df)

    # Add percentage labels
    for p in ax.patches:
        count = p.get_height()
        percentage = f"{100 * count / total:.1f}%"
        x = p.get_x() + p.get_width() / 2
        y = count

        ax.text(x, y + total * 0.005, percentage,
                ha="center", va="bottom", fontsize=10)

    # Add headroom so labels don't overflow
    ymax = max(p.get_height() for p in ax.patches)
    ax.set_ylim(0, ymax * 1.15)

# Perform ablation study

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

TARGET = TARGETS[0]   # or TARGETS[1]
RANDOM_STATE = 42
TEST_SIZE = 0.2

df = df.reset_index(drop=True)

X = df[ALL_FEATURES]
y = df[TARGET]

# Encode labels for XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
print(y.value_counts())

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# --- Load or create train/test split ---
try:
    train_idx = np.load(f"train_idx{len(df)}.npy")
    test_idx = np.load(f"test_idx{len(df)}.npy")
    print("Loaded saved train/test split")
except FileNotFoundError:
    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    )
    np.save(f"train_idx{len(df)}.npy", train_idx)
    np.save(f"test_idx{len(df)}.npy", test_idx)
    print("Created and saved train/test split")

# --- Feature sets ---
feature_sets = {
    "Static": STATIC_FEATURES,
    "Static + Graph": STATIC_FEATURES + GRAPH_FEATURES,
    "Static + SQL": STATIC_FEATURES + SQL_FEATURES,
    "Static + Dynamic": STATIC_FEATURES + DYNAMIC_FEATURES,
    "Static + Graph + SQL": STATIC_FEATURES + GRAPH_FEATURES + SQL_FEATURES,
    "Static + Graph + SQL + Dynamic": STATIC_FEATURES + GRAPH_FEATURES + SQL_FEATURES + DYNAMIC_FEATURES,
    "SQL": SQL_FEATURES,
}

# --- Label encoding and class weights ---
le = LabelEncoder()
y_encoded_le = le.fit_transform(y_encoded)  # fit on full dataset

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_encoded_le[train_idx]),
    y=y_encoded_le[train_idx]
)
sample_weights = np.array([class_weights[c] for c in y_encoded_le[train_idx]])

# --- Model creation ---
def create_model():
    return XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

# --- Ablation loop ---
results = []

for setting_name, cols in feature_sets.items():
    cols = [c for c in cols if c in df.columns]

    print(f"Running: {setting_name}")

    X_train = df.iloc[train_idx][cols]
    X_test = df.iloc[test_idx][cols]
    y_train = y_encoded_le[train_idx]
    y_test = y_encoded_le[test_idx]

    model = create_model()
    model.fit(X_train, y_train, sample_weight=sample_weights)

    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    results.append({
        "Features": setting_name,
        "Num Features": len(cols),
        "Accuracy": acc,
        "F1": f1,
        "ROC-AUC": auc
    })

# --- Save results ---
results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print("\nAblation Results:\n")
print(results_df)

results_df.to_csv(f"ablation_results_{TARGET}_{len(df)}.csv", index=False)
print(f"\nSaved results to ablation_results_{TARGET}_{len(df)}.csv")